# 00 · Explore the Zarr validation cubes

The validation cubes v2 live as **one Zarr store per site and year**,
`cubes/<site>/<year>.zarr`, written by `experiments/download_cube_zarr.py`
(bands `B02 B03 B04 B08 B8A B11 B12 SCL CLD` of SENTINEL2_L2A, 10 m, the grid
of the existing netCDF cubes; 20 m bands brought to 10 m by nearest
neighbour). `experiments.zarr_cubes.open_cube(site, start, end)` opens the
yearly stores a window needs, lazily, and selects the dates: the temporal
window of a validation run is a setting, and any window is a cheap selection.

This notebook opens a window and looks at it: dates and bands, an RGB, the
water indices (NIR, NDWI, MNDWI, AWEI, WRI) of one date, the SCL and CLD cloud
masks, the time series of one pixel and a quick-look grid of several dates.
Nothing is loaded whole: every cell reads one date, one pixel or a decimated
view. matplotlib only.

* Values are **raw int16** as the backend delivered them (reflectance × 10000
  for the spectral bands, class codes for SCL, cloud probability in % for
  CLD), fill **-32768**; `zc.as_float` turns a band into float32 with NaN.
* `END` is **exclusive** (as openEO and the yearly windows `[YYYY-01-01, YYYY+1-01-01)`).
* The format test store (`download_cube_zarr --test`) is opened with
  `ROOT = "cubes/_test"`, `START, END = "2025-06-01", "2025-06-16"`.

In [ ]:
# ── settings ─────────────────────────────────────────────────────────────────
SITE = "villaviciosa"                    # villaviciosa | ferrol | escalda | wadden | ems
START, END = "2023-01-01", "2026-01-01"  # END exclusive: 2023, 2024 and 2025
ROOT = None                              # None: $ZARR_CUBES_ROOT or <repo>/cubes ("cubes/_test": the --test store)
DATE = None                              # 'YYYY-MM-DD' for the single-date maps; None: the clearest date
PIXEL_LONLAT = (-5.395, 43.52)           # time-series pixel (lon, lat); None or outside the grid: grid centre
N_QUICKLOOK = 12                         # dates in the quick-look grid
CLD_MAX = 20                             # CLD (%) above which a pixel counts as cloudy
MAX_MAP_PX = 1500                        # maps are decimated to at most this many pixels per side

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments" / "zarr_cubes.py").is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from experiments import zarr_cubes as zc

print("cubes root:", zc.cubes_root(ROOT))
print("complete stores:", zc.available(ROOT))

## 1 · Open a window

`open_cube` opens only the years the window touches, checks they share the
grid and concatenates them along `t`; nothing is read yet (dask arrays).

In [ ]:
ds = zc.open_cube(SITE, START, END, root=ROOT)
bands = zc.band_names(ds)
dates = zc.dates(ds)
T, H, W = ds.sizes["t"], ds.sizes["y"], ds.sizes["x"]
g = zc.grid_info(ds)
print(f"{SITE}: {T} dates, {H} x {W} px at {g['dx']:g} m, EPSG:{g['epsg']}, years {ds.attrs['years']}")
print(f"window [{ds.attrs['window_start']}, {ds.attrs['window_end_exclusive']})  "
      f"first {dates[0] if dates else '-'}  last {dates[-1] if dates else '-'}")
print("bands:", bands)
print(pd.DataFrame({b: {"dtype": str(ds[b].dtype), "fill": zc.fill_of(ds[b]),
                        "chunks (t, y, x)": tuple(c[0] for c in ds[b].chunks),
                        "native m": ds[b].attrs.get("native_resolution_m"),
                        "resampling": ds[b].attrs.get("resampling", "")[:40]} for b in bands}).T)
print("\nprovenance per year:")
for s in json.loads(ds.attrs["sources"]):
    print(f"  {s['year']}: job {s['job_id']}, processed {s['processing_date']}  ({s['store']})")
ds

In [ ]:
# Another window is just another selection over the same stores (nothing is rewritten)
one_year = zc.open_cube(SITE, START, pd.Timestamp(START) + pd.DateOffset(years=1), root=ROOT,
                        bands=["B03", "B08", "SCL"])
print(f"first year only: {one_year.sizes['t']} dates, bands {zc.band_names(one_year)}")
one_year.close()

## 2 · Dates

Dates per month, and the clear-sky share of each date measured on a central
window of at most 512 × 512 px (SCL classes 4–7 and CLD ≤ `CLD_MAX`). The
central window keeps this cheap: only those chunks are read.

In [ ]:
t = pd.DatetimeIndex(ds["t"].values)
per_month = pd.crosstab(t.year, t.month).reindex(columns=range(1, 13), fill_value=0)
per_month.index.name, per_month.columns.name = "year", "month"
print(per_month.assign(total=per_month.sum(axis=1)))

cy, cx = H // 2, W // 2
win = dict(y=slice(max(0, cy - 256), cy + 256), x=slice(max(0, cx - 256), cx + 256))
scl_c = ds["SCL"].isel(**win)
cld_c = ds["CLD"].isel(**win)
valid_c = scl_c != zc.fill_of(ds["SCL"])
clear_c = zc.clear_mask(scl_c) & (cld_c <= CLD_MAX)
clear_share = (clear_c.sum(("y", "x")) / valid_c.sum(("y", "x")).clip(min=1)).compute().values
valid_share = valid_c.mean(("y", "x")).compute().values

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(t, clear_share * 100, width=3, color="tab:blue", label="clear (SCL 4-7 and CLD <= %d %%)" % CLD_MAX)
ax.plot(t, valid_share * 100, ".", color="0.5", ms=3, label="pixels with data")
ax.set_ylabel("% of the central window")
ax.set_ylim(0, 105)
ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.set_title(f"{SITE}: {T} dates in [{START}, {END})")
plt.show()

## 3 · One date: RGB and water indices

All bands of one date, decimated for display. Indices are computed on
reflectance (DN / 10000) with NaN where any band has no data:
NDWI = (B03 − B08)/(B03 + B08), MNDWI = (B03 − B11)/(B03 + B11),
AWEI_nsh = 4 (B03 − B11) − (0.25 B08 + 2.75 B12), WRI = (B03 + B04)/(B08 + B11).

In [ ]:
it = dates.index(DATE) if DATE else int(np.nanargmax(clear_share))
DATE_SHOWN = dates[it]
step = max(1, int(np.ceil(max(H, W) / MAX_MAP_PX)))
scene = ds[bands].isel(t=it, y=slice(None, None, step), x=slice(None, None, step)).compute()
refl = {b: zc.as_float(scene[b]).values / 10000.0 for b in bands if b not in ("SCL", "CLD")}
x_km = (scene["x"].values - scene["x"].values[0]) / 1000.0
y_km = (scene["y"].values - scene["y"].values[-1]) / 1000.0
extent = [x_km[0], x_km[-1], y_km[-1], y_km[0]]
print(f"date {DATE_SHOWN} (clear share of the central window {clear_share[it] * 100:.0f} %), "
      f"display step {step} px")


def stretch(rgb, lo_hi=None, pct=(2, 98)):
    ok = np.isfinite(rgb).all(axis=-1)
    if lo_hi is None:
        lo_hi = np.percentile(rgb[ok], pct) if ok.any() else (0.0, 1.0)
    lo, hi = lo_hi
    out = np.clip((rgb - lo) / max(hi - lo, 1e-6), 0, 1)
    out[~ok] = 1.0
    return out, (lo, hi)


rgb, _ = stretch(np.dstack([refl["B04"], refl["B03"], refl["B02"]]))
G, R, N, S1, S2 = refl["B03"], refl["B04"], refl["B08"], refl["B11"], refl["B12"]
with np.errstate(divide="ignore", invalid="ignore"):
    idx = {"NIR (B08)": (N, "magma", (0, np.nanpercentile(N, 99))),
           "NDWI": ((G - N) / (G + N), "RdBu", (-1, 1)),
           "MNDWI": ((G - S1) / (G + S1), "RdBu", (-1, 1)),
           "AWEI_nsh": (4 * (G - S1) - (0.25 * N + 2.75 * S2), "RdBu", None),
           "WRI": ((G + R) / (N + S1), "viridis", (0, 2))}

fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
axes[0, 0].imshow(rgb, extent=extent)
axes[0, 0].set_title(f"RGB (B04, B03, B02), 2-98 % stretch\n{SITE} {DATE_SHOWN}")
for ax, (name, (arr, cmap, lim)) in zip(axes.flat[1:], idx.items()):
    if lim is None:
        m = np.nanpercentile(np.abs(arr), 98)
        lim = (-m, m)
    im = ax.imshow(arr, cmap=cmap, vmin=lim[0], vmax=lim[1], extent=extent, interpolation="nearest")
    ax.set_title(name)
    fig.colorbar(im, ax=ax, shrink=0.8)
for ax in axes.flat:
    ax.set_xlabel("km")
    ax.set_ylabel("km")
plt.show()

## 4 · Cloud filters: SCL and CLD

SCL is the Sen2Cor scene classification, CLD the Sen2Cor cloud probability
(%), both 20 m products brought to 10 m by nearest neighbour. The last panel
shows where the two filters agree.

In [ ]:
scl = scene["SCL"].values
cld = scene["CLD"].values.astype("float32")
scl_fill, cld_fill = zc.fill_of(scene["SCL"]), zc.fill_of(scene["CLD"])
nodata = (scl == scl_fill) | (cld == cld_fill) | (scl == 0)
cld[cld == cld_fill] = np.nan

codes = sorted(zc.SCL_CLASSES)
scl_cmap = ListedColormap([zc.SCL_CLASSES[c][1] for c in codes])
scl_norm = BoundaryNorm(np.arange(-0.5, len(codes) + 0.5), scl_cmap.N)
scl_show = np.ma.masked_where(scl == scl_fill, scl)

clear_scl = np.isin(scl, zc.CLEAR_SCL)
clear_cld = cld <= CLD_MAX
agree = np.full(scl.shape, 0, "int8")              # 0 no data
agree[~nodata & clear_scl & clear_cld] = 1         # clear by both
agree[~nodata & ~clear_scl & clear_cld] = 2        # cloudy by SCL only
agree[~nodata & clear_scl & ~clear_cld] = 3        # cloudy by CLD only
agree[~nodata & ~clear_scl & ~clear_cld] = 4       # cloudy by both
agree_cols = ["#000000", "#9ecae1", "#fdae6b", "#a1d99b", "#636363"]
agree_lab = ["no data", "clear (both)", f"cloudy: SCL only", f"cloudy: CLD > {CLD_MAX} % only", "cloudy (both)"]

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), constrained_layout=True)
axes[0].imshow(scl_show, cmap=scl_cmap, norm=scl_norm, extent=extent, interpolation="nearest")
present = [c for c in codes if (scl == c).any()]
axes[0].legend(handles=[Patch(facecolor=zc.SCL_CLASSES[c][1], edgecolor="0.4", label=f"{c} {zc.SCL_CLASSES[c][0]}") for c in present],
               fontsize=7, loc="lower left", framealpha=0.8)
axes[0].set_title(f"SCL {DATE_SHOWN}")
im = axes[1].imshow(cld, cmap="magma", vmin=0, vmax=100, extent=extent, interpolation="nearest")
fig.colorbar(im, ax=axes[1], shrink=0.8, label="%")
axes[1].set_title("CLD cloud probability")
axes[2].imshow(agree, cmap=ListedColormap(agree_cols), vmin=-0.5, vmax=4.5, extent=extent, interpolation="nearest")
axes[2].legend(handles=[Patch(facecolor=c, edgecolor="0.4", label=l) for c, l in zip(agree_cols, agree_lab)], fontsize=7,
               loc="lower left", framealpha=0.8)
axes[2].set_title("SCL clear classes 4-7 vs CLD threshold")
plt.show()
ok = ~nodata
print(f"of the pixels with data: clear by SCL {clear_scl[ok].mean() * 100:.1f} %, by CLD {clear_cld[ok].mean() * 100:.1f} %, "
      f"filters disagree on {((clear_scl != clear_cld) & ok).sum() / max(ok.sum(), 1) * 100:.1f} %")

## 5 · Time series of one pixel

All dates of the window at one pixel (one column of chunks per band is read).
Filled markers: clear by SCL and CLD; hollow: flagged by either filter.

In [ ]:
xs, ys = ds["x"].values, ds["y"].values
X, Y = float(xs[len(xs) // 2]), float(ys[len(ys) // 2])
where = "grid centre"
if PIXEL_LONLAT is not None:
    from pyproj import Transformer

    tf = Transformer.from_crs("EPSG:4326", ds["crs"].attrs["crs_wkt"], always_xy=True)
    px_, py_ = tf.transform(*PIXEL_LONLAT)
    if xs.min() <= px_ <= xs.max() and ys.min() <= py_ <= ys.max():
        X, Y, where = px_, py_, f"lon/lat {PIXEL_LONLAT}"
    else:
        print(f"{PIXEL_LONLAT} is outside the grid: using its centre")
pix = ds[bands].sel(x=X, y=Y, method="nearest").compute()
row, col = int(np.argmin(np.abs(ys - Y))), int(np.argmin(np.abs(xs - X)))
print(f"pixel row {row} col {col} (x {float(pix['x']):.0f}, y {float(pix['y']):.0f}; {where})")

tt = pd.DatetimeIndex(pix["t"].values)
f = {b: zc.as_float(pix[b]).values for b in bands}
with np.errstate(divide="ignore", invalid="ignore"):
    ndwi = (f["B03"] - f["B08"]) / (f["B03"] + f["B08"])
    mndwi = (f["B03"] - f["B11"]) / (f["B03"] + f["B11"])
clear = np.isin(f["SCL"], zc.CLEAR_SCL) & (f["CLD"] <= CLD_MAX)

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True, constrained_layout=True)
for arr, name, colr in ((ndwi, "NDWI", "tab:blue"), (mndwi, "MNDWI", "tab:green")):
    axes[0].plot(tt[clear], arr[clear], "o", ms=4, color=colr, label=f"{name} (clear)")
    axes[0].plot(tt[~clear], arr[~clear], "o", ms=4, mfc="none", color=colr, alpha=0.5, label=f"{name} (flagged)")
axes[0].axhline(0, color="0.6", lw=0.8)
axes[0].set_ylabel("index")
axes[0].legend(fontsize=8, ncol=4, frameon=False)
for b, colr in (("B08", "tab:red"), ("B11", "tab:purple"), ("B03", "tab:green")):
    axes[1].plot(tt, f[b] / 10000.0, ".-", lw=0.5, ms=4, color=colr, label=b)
axes[1].set_ylabel("DN / 10000")
axes[1].legend(fontsize=8, ncol=3, frameon=False)
sc = f["SCL"]
okk = np.isfinite(sc)
axes[2].scatter(tt[okk], sc[okk], c=[zc.SCL_CLASSES.get(int(v), ("", "#ff00ff"))[1] for v in sc[okk]],
                edgecolors="0.3", linewidths=0.3, s=25)
axes[2].set_yticks(codes)
axes[2].set_yticklabels([f"{c} {zc.SCL_CLASSES[c][0]}" for c in codes], fontsize=7)
ax2 = axes[2].twinx()
ax2.plot(tt, f["CLD"], "k.", ms=3, alpha=0.6)
ax2.set_ylabel("CLD %")
ax2.set_ylim(-5, 105)
axes[2].set_title("SCL class (colour) and CLD (black dots)", fontsize=9)
fig.suptitle(f"{SITE}, pixel ({row}, {col}): {clear.sum()} clear of {len(tt)} dates")
plt.show()

## 6 · Quick-look grid

`N_QUICKLOOK` dates spread over the window (the clearest date of each of
`N_QUICKLOOK` equal time bins), RGB at a common stretch so dates compare.

In [ ]:
bins = np.array_split(np.arange(T), min(N_QUICKLOOK, T))
pick = sorted({int(b[np.nanargmax(clear_share[b])]) for b in bins if len(b)})
qstep = max(1, int(np.ceil(max(H, W) / 400)))
ql = ds[["B04", "B03", "B02"]].isel(t=pick, y=slice(None, None, qstep), x=slice(None, None, qstep)).compute()
stack = np.stack([zc.as_float(ql[b]).values / 10000.0 for b in ("B04", "B03", "B02")], axis=-1)
_, common = stretch(stack.reshape(-1, 1, 3))
ncol = 4
nrow = int(np.ceil(len(pick) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.6 * nrow), squeeze=False, constrained_layout=True)
for k, ax in enumerate(axes.flat):
    ax.set_axis_off()
    if k < len(pick):
        img, _ = stretch(stack[k], common)
        ax.imshow(img, interpolation="nearest")
        ax.set_title(f"{dates[pick[k]]}  clear {clear_share[pick[k]] * 100:.0f} %", fontsize=9)
fig.suptitle(f"{SITE}: {len(pick)} dates of [{START}, {END}), step {qstep} px")
plt.show()

In [ ]:
ds.close()